# 2 - Selection

Select 60 objects: top-10 objects with the densest average cadence in each band (u/g/r/i/z/y) 

Average cadence is the inverse of total_length_of_lightcurve_in_days/number_of_observations

In [1]:
import lsdb
import pandas as pd

In [2]:
# Import and set up dask

from dask.distributed import Client
import warnings

# Suppress the port-in-use UserWarning
warnings.filterwarnings("ignore", message="Port 8787 is already in use")

# Raise the log level for distributed so INFO messages are hidden
logging.getLogger("distributed").setLevel(logging.WARNING)

# Make a dask client
client = Client(
    n_workers=4,
    memory_limit="4GB",
)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: proxy/8787/status,
Dashboard: proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 14.90 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39609,Workers: 0
Dashboard: proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:41807,Total threads: 1
Dashboard: proxy/32797/status,Memory: 3.73 GiB
Nanny: tcp://127.0.0.1:41683,


In [3]:
# Get dask dashboard link (if running on USDF RSP)

base_url = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '')
dashboard_url = f"https://usdf-rsp.slac.stanford.edu{base_url}proxy/8787/status"
print(f"Dask Dashboard: {dashboard_url}")

Dask Dashboard: https://usdf-rsp.slac.stanford.edu/nb/user/olynn/proxy/8787/status


## Import catalog

In [4]:
gxd = lsdb.open_catalog("xmatched_cats/ga_x_dp2_full")
gxd

,_RAJ2000,_DEJ2000,diaObjectId,ra,dec,diaObjectForcedSource,_dist_arcsec
npartitions=1982,,,,,,,
"Order: 7, Pixel: 68159",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",double[pyarrow]
"Order: 4, Pixel: 1065",...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12240",...,...,...,...,...,...,...
"Order: 5, Pixel: 12256",...,...,...,...,...,...,...


## Calculate Average Cadence

We'll add columns to store our per-band average cadence calculations

### Plan  map_rows

In [5]:
df = gxd.head(5)

In [6]:
df.iloc[0]["diaObjectForcedSource"]

,parentObjectId,coord_ra,coord_dec,visit,detector,band,psfFlux,psfFluxErr,psfFlux_flag,psfDiffFlux,...,pixelFlags_saturated,pixelFlags_saturatedCenter,pixelFlags_suspect,pixelFlags_suspectCenter,invalidPsfFlag,tract,patch,psfMag,psfMagErr,midpointMjdTai
0,0,342.989192,-19.60707,2025071700451,178,r,81670.203125,365.009857,False,0.000000,...,False,False,False,False,False,6358,73,19.119841,0.004853,60874.245523
1,0,342.989192,-19.60707,2025071700490,178,i,81169.234375,471.199432,False,10570.016602,...,False,False,False,False,False,6358,73,19.126522,0.006303,60874.266559
2,0,342.989192,-19.60707,2025111400142,11,g,63146.703125,244.775803,False,-1718.788574,...,False,False,False,False,False,6358,73,19.399124,0.004209,60994.110840
3,0,342.989192,-19.60707,2025111400246,21,i,74506.968750,419.456207,False,8382.705078,...,False,False,False,False,False,6358,73,19.219507,0.006112,60994.169144
4,0,342.989192,-19.60707,2025111600123,31,g,65056.226562,271.612671,False,-581.250122,...,False,False,False,False,False,6358,73,19.366777,0.004533,60996.085622
5,0,342.989192,-19.60707,2025111600237,28,i,78784.273438,619.326660,False,1223.831421,...,False,False,False,False,False,6358,73,19.158901,0.008535,60996.155622
6,0,342.989192,-19.60707,2025111800099,33,g,63813.558594,301.951996,False,-2610.370605,...,False,False,False,False,False,6358,73,19.387718,0.005138,60998.158966
7,0,342.989192,-19.60707,2025111900082,21,i,70136.093750,357.216766,False,4207.729492,...,False,False,False,False,False,6358,73,19.285147,0.005530,60999.100587
8,0,342.989192,-19.60707,2025112100092,10,g,63366.074219,252.065765,False,-3001.543945,...,False,False,False,False,False,6358,73,19.395357,0.004319,61001.085527
9,0,342.989192,-19.60707,2025112100197,11,i,74598.117188,467.644714,False,4383.223633,...,False,False,False,False,False,6358,73,19.218180,0.006806,61001.166060


In [7]:
def count_total_days_and_nobs(row):
    res = []
    # Get total days
    days = (
        row["diaObjectForcedSource"]["midpointMjdTai"].max() - 
        row["diaObjectForcedSource"]["midpointMjdTai"].min()
    )
    res.append(days)
    # Get nobs
    for band in "ugrizy":
        res.append(
            (row["diaObjectForcedSource"]["band"] == band).sum()
        )
    # Do the final math for average cadence
    for band in ['u', 'g', 'r', 'i', 'z', 'y']:
        nobs = (row["diaObjectForcedSource"]["band"] == band).sum()
        res.append(nobs/days)
    return res

gxd_added_cols = gxd.map_rows(
    count_total_days_and_nobs,  # our user-defined fn
    columns=["diaObjectForcedSource"],  # col(s) to pass to fn
    output_names=[
        "total_days", 
        "nobs_u", 
        "nobs_g", 
        "nobs_r", 
        "nobs_i", 
        "nobs_z", 
        "nobs_y",
        "av_cad_u",
        "av_cad_g",
        "av_cad_r",
        "av_cad_i",
        "av_cad_z",
        "av_cad_y",
    ],  # new col(s)
    meta={
        "total_days": float,
        "nobs_u": int,
        "nobs_g": int,
        "nobs_r": int,
        "nobs_i": int,
        "nobs_z": int,
        "nobs_y": int,
        "av_cad_u": float,
        "av_cad_g": float,
        "av_cad_r": float,
        "av_cad_i": float,
        "av_cad_z": float,
        "av_cad_y": float,
    },  # describe for dask
    append_columns=False,  # keep this false while we're just previewing
)

gxd_added_cols.head(5)

,total_days,nobs_u,nobs_g,nobs_r,nobs_i,nobs_z,nobs_y,av_cad_u,av_cad_g,av_cad_r,av_cad_i,av_cad_z,av_cad_y
_healpix_29,,,,,,,,,,,,,
1199082000028818063,126.920537,0,4,1,5,0,0,0.0,0.031516,0.007879,0.039395,0.000000,0.0
1199799084675512657,7.055220,0,2,0,2,0,0,0.0,0.283478,0.000000,0.283478,0.000000,0.0
1199815675876963592,134.001233,0,4,0,4,2,0,0.0,0.029850,0.000000,0.029850,0.014925,0.0
1199835500363950558,134.001233,0,3,1,4,2,0,0.0,0.022388,0.007463,0.029850,0.014925,0.0
1199852887212822005,142.801858,0,5,2,3,2,0,0.0,0.035014,0.014005,0.021008,0.014005,0.0


### Run map_rows

In [8]:
def calculate_average_cadence(row):
    res = []
    # Get total days
    days = (
        row["diaObjectForcedSource"]["midpointMjdTai"].max() - 
        row["diaObjectForcedSource"]["midpointMjdTai"].min()
    )
    # Divide nobs by days for average cadence
    for band in ["u", "g", "r", "i", "z", "y"]:
        nobs = (row["diaObjectForcedSource"]["band"] == band).sum()
        res.append(0 if days == 0 else nobs/days)
    return res
    

gxd_w_cadence = gxd.map_rows(
    calculate_average_cadence,  # our user-defined fn
    columns=["diaObjectForcedSource"],  # col(s) to pass to fn
    output_names=[
        "av_cad_u",
        "av_cad_g",
        "av_cad_r",
        "av_cad_i",
        "av_cad_z",
        "av_cad_y",
    ],  # new col(s)
    meta={
        "av_cad_u": float,
        "av_cad_g": float,
        "av_cad_r": float,
        "av_cad_i": float,
        "av_cad_z": float,
        "av_cad_y": float,
    },  # describe for dask
    append_columns=True,  # original cat + added cols
)


gxd_w_cadence.head(5)

_RAJ2000   _DEJ2000        diaObjectId          ra  \
_healpix_29                                                                 
1199082000028818063  342.989184 -19.607069  43696859830550585  342.989192   
1199799084675512657  343.535528 -19.751029  43703456900317185  343.535549   
1199815675876963592  343.918047 -19.548181  43704693850898496  343.918079   
1199835500363950558  343.489634 -19.444504  43697340866887709  343.489828   
1199852887212822005  343.664362 -19.045443  45288128033849399  343.664341   

                           dec  \
_healpix_29                      
1199082000028818063  -19.60707   
1199799084675512657 -19.751035   
1199815675876963592 -19.548171   
1199835500363950558 -19.444423   
1199852887212822005 -19.045427   

                                                 diaObjectForcedSource  \
_healpix_29                                                              
1199082000028818063  [{parentObjectId: 0, coord_ra: 342.989192, coo...   
1199799084675512657  [{parentObjectId: 0, coord_ra: 343.535549, coo...   
1199815675876963592  [{parentObjectId: 0, coord_ra: 343.918079, coo...   
1199835500363950558  [{parentObjectId: 0, coord_ra: 343.489828, coo...   
1199852887212822005  [{parentObjectId: 0, coord_ra: 343.664341, coo...   

                     _dist_arcsec  av_cad_u  av_cad_g  av_cad_r  av_cad_i  \
_healpix_29                                                                 
1199082000028818063      0.025907       0.0  0.031516  0.007879  0.039395   
1199799084675512657      0.074434       0.0  0.283478  0.000000  0.283478   
1199815675876963592      0.113474       0.0  0.029850  0.000000  0.029850   
1199835500363950558      0.719588       0.0  0.022388  0.007463  0.029850   
1199852887212822005      0.091714       0.0  0.035014  0.014005  0.021008   

                     av_cad_z  av_cad_y  
_healpix_29                              
1199082000028818063  0.000000       0.0  
1199799084675512657  0.000000       0.0  
1199815675876963592  0.014925       0.0  
1199835500363950558  0.014925       0.0  
1199852887212822005  0.014005       0.0

## Select by cadence

Select the top 10 objects with the densest average in each band

### Plan map partitions: 
- get largest for each data frame,
- then put them together,
- then get the 10th largest value
- then use that as a minimum cutoff for a query

In [14]:
# Use map partitions to get the "best representative" from each partition

band = "z"
av_cad_col = f"av_cad_{band}"

def get_best_cadence(df):
    mini_df = df[["diaObjectId", av_cad_col]]
    mini_df = mini_df.nlargest(1, av_cad_col)
    # Throw out trash data (where av cadence is ~0 - we see this a lot in z and y)
    if len(mini_df) > 0 and abs(mini_df[av_cad_col].iloc[0]) < 1e-10:
        return None
    return mini_df

# Think the following is unnecessary in this case after all
# output_meta = pd.DataFrame([{ "diaObjectId": 0, col: 0.0, }])

best_partition_representatives = gxd_w_cadence.map_partitions(
    get_best_cadence,
    include_pixel=False,
    #meta=output_meta,
)

bpr_df = best_partition_representatives.compute()
bpr_df

,diaObjectId,av_cad_z
_healpix_29,,
1199861098219892755,45287097241698355,22.669269
1200269342220859970,42117342657773635,0.143298
...,...,...
3445255975108960306,50009430963519952,0.055127
3449774236355820968,49963595072536821,0.110527


In [18]:
# Get the 10 largest of the per-partition representatives

top_10_reps = bpr_df.nlargest(10, av_cad_col)
sorted_df = top_10_reps.sort_values(by=av_cad_col)
sorted_df

,diaObjectId,av_cad_z
_healpix_29,,
2489855296153773325,19771624249098827,0.667168
2490363097202629456,19767157483110602,0.671275
2490161772572968228,19765233337761919,0.673125
2490241912783579752,19773617113924318,0.679082
2041921839966981390,43464587999186109,0.722570
3442713336320275306,49992182374858788,2.011843
3414404163346367415,42023884169412636,6.991297
1199861098219892755,45287097241698355,22.669269
3427676269200558329,41981965288603756,29.998208


In [19]:
gxd_w_cadence

,_RAJ2000,_DEJ2000,diaObjectId,ra,dec,diaObjectForcedSource,_dist_arcsec,av_cad_u,av_cad_g,av_cad_r,av_cad_i,av_cad_z,av_cad_y
npartitions=1982,,,,,,,,,,,,,
"Order: 7, Pixel: 68159",double[pyarrow],double[pyarrow],int64[pyarrow],double[pyarrow],double[pyarrow],"nested<parentObjectId: [int64], coord_ra: [dou...",double[pyarrow],float64,float64,float64,float64,float64,float64
"Order: 4, Pixel: 1065",...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12240",...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 5, Pixel: 12256",...,...,...,...,...,...,...,...,...,...,...,...,...


In [22]:
#%%time

# If we select the 10th largest, we know there are at least 9 other rows larger
# It's a reasonable lower bound from which to run our query

# If we had a truly massive catalog and wanted to avoid running a query, we could assume
# that the only rows that could possibly be larger would be in the other 9 partitions, 
# and just search them. As it stands, the query is fast enough that it's perfectly sufficient.

# (At least, usually. Took forever running it Thurs evening for some reason, but typically it's ~30 sec)

min_cutoff = sorted_df[av_cad_col].iloc[0]
print("Minimum cutoff:", min_cutoff)

gxd_w_cadence.query(f"{av_cad_col} >= {min_cutoff}")
ndf = gxd_w_cadence.compute()
print("Rows above minimum cutoff:", len(df))

top_rows = ndf.nlargest(10, av_cad_col)
top_rows

Minimum cutoff: 0.6671680479907229
Rows above minimum cutoff: 5


_RAJ2000   _DEJ2000        diaObjectId          ra  \
_healpix_29                                                                 
3414360272518473009  319.106193 -22.187857  42015981429588060  319.106193   
3427676269200558329  310.198468 -22.175824  41981965288603756  310.198471   
1199861098219892755  344.574552 -19.237487  45287097241698355   344.57444   
3414404163346367415  319.701375 -21.992483  42023884169412636   319.70135   
3442713336320275306  316.257645 -13.577976  49992182374858788  316.257645   
2041921839966981390  289.436747  -19.35878  43464587999186109  289.436742   
2490241912783579752   10.248385 -43.960197  19773617113924318    10.24838   
2490161772572968228    8.638204 -44.245255  19765233337761919    8.638227   
2490363097202629456    9.149776 -43.776705  19767157483110602     9.14974   
2489855296153773325   10.070385 -44.432293  19771624249098827   10.070375   

                           dec  \
_healpix_29                      
3414360272518473009 -22.187857   
3427676269200558329 -22.175814   
1199861098219892755 -19.237682   
3414404163346367415 -21.992488   
3442713336320275306 -13.577978   
2041921839966981390 -19.358786   
2490241912783579752  -43.96019   
2490161772572968228 -44.245241   
2490363097202629456  -43.77671   
2489855296153773325 -44.432293   

                                                 diaObjectForcedSource  \
_healpix_29                                                              
3414360272518473009  [{parentObjectId: 0, coord_ra: 319.106193, coo...   
3427676269200558329  [{parentObjectId: 0, coord_ra: 310.198471, coo...   
1199861098219892755  [{parentObjectId: 0, coord_ra: 344.57444, coor...   
3414404163346367415  [{parentObjectId: 0, coord_ra: 319.70135, coor...   
3442713336320275306  [{parentObjectId: 0, coord_ra: 316.257645, coo...   
2041921839966981390  [{parentObjectId: 0, coord_ra: 289.436742, coo...   
2490241912783579752  [{parentObjectId: 0, coord_ra: 10.24838, coord...   
2490161772572968228  [{parentObjectId: 0, coord_ra: 8.638227, coord...   
2490363097202629456  [{parentObjectId: 0, coord_ra: 9.14974, coord_...   
2489855296153773325  [{parentObjectId: 0, coord_ra: 10.070375, coor...   

                     _dist_arcsec  av_cad_u  av_cad_g  av_cad_r   av_cad_i  \
_healpix_29                                                                  
3414360272518473009      0.001289  0.000000  0.000000  0.000000  46.998088   
3427676269200558329      0.035152  0.000000  0.000000  0.000000  29.998208   
1199861098219892755      0.797943  0.000000  0.000000  0.000000   0.000000   
3414404163346367415      0.088105  0.000000  0.000000  0.000000   3.495648   
3442713336320275306      0.006558  0.000000  0.000000  0.000000   0.000000   
2041921839966981390      0.026334  0.278887  0.456360  0.633833   0.988780   
2490241912783579752      0.029297  0.285929  0.536117  0.548031   0.994795   
2490161772572968228       0.07734  0.268059  0.559945  0.506333   1.042450   
2490363097202629456      0.095687  0.227754  0.563392  0.527430   0.970951   
2489855296153773325      0.026146  0.220404  0.565901  0.512290   1.018623   

                      av_cad_z  av_cad_y  
_healpix_29                               
3414360272518473009  46.998088  0.000000  
3427676269200558329  29.998208  0.000000  
1199861098219892755  22.669269  0.000000  
3414404163346367415   6.991297  3.495648  
3442713336320275306   2.011843  1.005922  
2041921839966981390   0.722570  0.367623  
2490241912783579752   0.679082  0.154878  
2490161772572968228   0.673125  0.178706  
2490363097202629456   0.671275  0.167819  
2489855296153773325   0.667168  0.160835

In [23]:
# Drop observations that have NA values
# When checking for band z, it's only a couple rows, but it makes a huge difference later on!

print(top_rows["diaObjectForcedSource"].apply(len).to_list())

top_rows_drop_nested_na = top_rows.dropna(on_nested="diaObjectForcedSource")

print(top_rows_drop_nested_na["diaObjectForcedSource"].apply(len).to_list())


[2, 2, 2, 4, 3, 272, 537, 542, 522, 528]
[2, 2, 2, 4, 3, 272, 536, 542, 522, 526]


### Run map_partitions to generate top-10 catalog for each band

In [27]:
# Create function for what we did above

def get_top_10_rows(cat, band):
    av_cad_col = f"av_cad_{band}"
    
    # Use map partitions to get the "best representative" from each partition
    def get_best_cadence(df):
        mini_df = df[["diaObjectId", av_cad_col]]
        mini_df = mini_df.nlargest(1, av_cad_col)
        # Throw out trash data (where av cadence is ~0 - we see this a lot in z and y)
        if len(mini_df) > 0 and abs(mini_df[av_cad_col].iloc[0]) < 1e-10:
            return None
        return mini_df
    
    best_partition_representatives = cat.map_partitions(
        get_best_cadence,
        include_pixel=False,
    )
    bpr_df = best_partition_representatives.compute()

    # Get the 10 largest of the per-partition representatives
    top_10_reps = bpr_df.nlargest(10, av_cad_col)
    sorted_df = top_10_reps.sort_values(by=av_cad_col)

    # Use the 10th largest value as a minimum cutoff
    min_cutoff = sorted_df[av_cad_col].iloc[0]    
    df = gxd_w_cadence.query(f"{av_cad_col} >= {min_cutoff}").compute()

    # Then pick the 10 largest
    top_rows = df.nlargest(10, av_cad_col)

    # Drop NA observations
    top_rows_drop_nested_na = top_rows.dropna(on_nested="diaObjectForcedSource")

    return top_rows

In [28]:
# Generate top-10 catalogs for each band and save to disk

top_10_dfs = {}

for band in "ugrizy":
    print(f"Generating top-10 catalog for band {band}...")
    top_10_dfs[band] = get_top_10_rows(gxd_w_cadence, band)
    cat = lsdb.from_dataframe(top_10_dfs[band])
    cat.write_catalog(f"processed_catalogs/top_10_{band}", overwrite=True)

Generating top-10 catalog for band u...
Generating top-10 catalog for band g...
Generating top-10 catalog for band r...
Generating top-10 catalog for band i...
Generating top-10 catalog for band z...
Generating top-10 catalog for band y...
